[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pabanib/Clases_clasificacion/blob/main/06_clustering_teorico.ipynb#copy=true)


[![Open in Anaconda](https://static.anaconda.cloud/content/a22d04e8445b700f28937ab3231b8cded505d0395c63b7a269696722196d5415)](https://anaconda.com/api/nbserve/launch_notebook?nb_url=https%3A%2F%2Fanaconda.com%2Fapi%2Fprojects%2F6905e11d-e422-418c-ad09-bff2871f164e%2Ffiles%2F06_clustering_teorico.ipynb%3Fversion%3Da5e821a8-ad2b-497a-a589-b3a925f271a3)




# Modelos no supervisados: Clustering


Las técnicas de clustering consisten en un conjunto de metodologías que buscan obtener subgrupos en los datos. Dentro de cada subgrupo las observaciones deben ser similares y heterogeneas en comparación con elementos de otros grupos.

 Cuando se hace un análisis clustering se parte de la premisa de que los datos no se han distribuido de forma completamente aleatoria y por lo tanto existen patrones en ellos que nos pueden dar información. Son técnicas de aprendizaje no supervisado por el hecho de que no existe una variable output que tratamos de predecir.

La partición en grupos tiene que cumplir las siguientes condiciones. Sean  $C_1,...,C_k$ los cluster formados, estos deben cumplir:

 1. $C_1\cup C_2 \cup ...\cup C_k = \{1,...,n\} $ o sea cada observación pertenece a al menos uno de los  $K$  subgrupos.
 2. $C_i \cap C_j = \emptyset$ para todo $i \neq j$ . En otras palabras los agrupamientos no se superponen.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

# K-Means


La idea detrás de *K-means clustering* es que existe un buen agrupamiento si la disimilitud dentro de cada cluster es la menor posible. Entonces podemos plantear el problema cómo la minimización de un función dada de la siguiente forma:

\begin{equation}
min_{C_1,...,C_K} \left(\sum_{k=1}^{K} W\left(C_k\right)\right)
\end{equation}

en dónde $W$  es la función de disimilitud de intra-cluster. En palabras, se busca una partición de los datos en  $K$  grupos en dónde la suma total de las disimilitudes sea la menor posible.

La función de disimilitud puede definirse de muchas maneras, pero la más habitual es la **distancia Euclídea**.

\begin{equation}
W\left(C_k\right) = \frac{1}{\left|C_k\right|} \sum_{i,i'∈C_k}\sum_{j=1}^p \left(x_{ij}-\mu_{k}\right)^2
\end{equation}

Donde $\left|C_k\right|$  refleja el número de observaciones dentro del cluster $k$ y $\mu_{k}$ es el centroide del grupo $k$.

Encontrar una solución a ese problema es realmente un problema muy difícil ya que existen  $K^n$  formas de dividir las observaciones dentro de  $K$  grupos. Esto es un número muy grande aún cuando haya pocos datos.

Para resolverlo, entonces, existen algoritmos que llegan a soluciones locales.

1. Asignar aleatoriamente de 1 a  $K$  clusters a cada dato.
2. Iterar hasta que la asignación de clusters deje de cambiar:
    - Por cada cluster calcular el centroide.
    - Asignara a cada observación el cluster del centroide más cercano.


Tomemos un ejemplo sencillo para ver como agruparíamos.

Desde los datos de ejemplos del precio de de viviendas de California, queremos separar en dos grupos de regiones de acuerdo al valor promedio de las viviendas.


In [ ]:
url = "https://raw.githubusercontent.com/pabanib/dataframes/master/housing.csv"
house = pd.read_csv(url)#.sample(100)
house.drop(columns= 'ocean_proximity', inplace=True)
house.head()

## 1. Asignamos aleatoriamente un grupo a cada punto


### Con MatPlotLib


In [ ]:
cate = np.round(np.random.rand(len(house)),0)

plt.scatter(x = np.array(house.index), y = house.median_house_value, c = cate)


### Con Plotly


In [ ]:
px.scatter(x = np.array(house.index),
           y = house.median_house_value,
           color = cate,
           template = "gridon")

## 2 . Calculamos los centroides y asignamos a que grupo pertenece cada punto


In [ ]:
house.info()

In [ ]:
centroides = house.groupby(cate).mean()['median_house_value']
centroides

Asignamos cada punto al centroide más cercano.


### Con MatPlotLib


In [ ]:
ncate = []

for i in house.median_house_value:
  g0 = (i-centroides.iloc[0])**2    #distancia del punto i al centroide 0
  g1 = (i-centroides.iloc[1])**2    #distancia del punto i al centroide 1
  if g0 < g1:
    ncate.append(0)
  else:
    ncate.append(1)

plt.scatter(x = np.array(house.index), y = house.median_house_value, c = ncate)


### Con Plotly


In [ ]:
px.scatter(x = np.array(house.index),
           y = house.median_house_value,
           color = ncate,
           template = "gridon")

## 3 . Repetimos el paso 2


In [ ]:
centroides = house.groupby(ncate).mean()['median_house_value']
centroides

### Con MatPlotLib


In [ ]:
ncate = []

for i in house.median_house_value:
  g0 = (i-centroides.iloc[0])**2
  g1 = (i-centroides.iloc[1])**2
  if g0 < g1:
    ncate.append(0)
  else:
    ncate.append(1)

plt.scatter(x = np.array(house.index), y = house.median_house_value, c = ncate)

### Con Plotly


In [ ]:
px.scatter(x = np.array(house.index),
           y = house.median_house_value,
           color = ncate,
           template = "gridon")

In [ ]:
centroides = house.groupby(ncate).mean()['median_house_value']
centroides

¿Qué pasa ahora si queremos agrupar unidades pero con dos características? Por supuesto, si aplicamos la categorización anterior solo va tener en cuenta el valor medio de los hogares y no la antigüedad de la casa.


In [ ]:
house[['housing_median_age',
       'median_house_value']].plot(kind = 'scatter',
                                   x = 'housing_median_age',
                                   y = 'median_house_value',
                                   c = ncate)

El proceso, es exáctamente el mismo solo que ahora hay que tener en cuenta que los centroides se corresponden a dos variables y las distancias se deben calcular por cada uno de ellos.


In [ ]:
df = house[['housing_median_age','median_house_value']]
cate = np.round(np.random.rand(len(house)),0)   #asignamos aleatoriamente los valores
centroides = df.groupby(cate).mean()            #calculamos los centroides de acuerdo a los grupos aleatorios

ncate = []
columnas = df.columns

for ind in range(len(df)):
  g0 = 0
  g1 = 0
  for c in columnas:                            #calculamos las distancias por cada una de las columnas del dataframe
    i = df.iloc[ind]
    g0 += (i[c]-centroides.iloc[0][c])**2       #distancia de la columna "c" a su respectivo centroide
    g1 += (i[c]-centroides.iloc[1][c])**2
  if g0 < g1:
    ncate.append(0)
  else:
    ncate.append(1)


In [ ]:
color = []
for i in ncate:
  if i == 0:
    color.append('green')
  else:
    color.append('orange')

house[['housing_median_age','median_house_value']].plot(kind = 'scatter',x = 'housing_median_age', y = 'median_house_value', c = color)

El problema que estamos teniendo es de escalas. La antigüedad de la vivienda maneja valores muy menores al precio promedio de las viviendas y es por eso que esta variable prácticamente no influye en la asignación de grupos. Estos métodos necesitan si o sí el escalado de variables.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(house[['housing_median_age','median_house_value']])
X = pd.DataFrame(X, columns =['housing_median_age','median_house_value'] )
X.head()

In [ ]:
veces = 3
df = X
cate = np.round(np.random.rand(len(X)),0)
centroides = df.groupby(cate).mean()

for v in range(veces):
  ncate = []
  columnas = df.columns

  for ind in range(len(df)):
    g0 = 0
    g1 = 0
    for c in columnas:
      i = df.iloc[ind]
      g0 += (i[c]-centroides.iloc[0][c])**2
      g1 += (i[c]-centroides.iloc[1][c])**2
    if g0 < g1:
      ncate.append(0)
    else:
      ncate.append(1)


color = []
for i in ncate:
  if i == 0:
    color.append('green')
  else:
    color.append('orange')

house[['housing_median_age','median_house_value']].plot(kind = 'scatter',x = 'housing_median_age', y = 'median_house_value', c = color)

# Agglomerative clustering


Una de las desventejas de K-means es que hay que elegir el número de grupos antes de iniciar el algoritmo. Los agrupamientos jerárquicos permiten iniciar el método sin la necesidad de comprometerse con el número de grupos que buscamos. Además tiene otra ventaja, ofrecen una representacíon de tipo árbol de decisión que se denomina dendrograma.

El **dendrograma** se va formando con la unión de las observaciones que son parecidas entre sí. Se inicia de las más parecidas que hay, sería las ramas más bajas, luego se van asociando los diferentes grupos a medida que sean más parecidos. Así hasta unir a todas las observaciones en un solo árbol.


![Diagrama de clustering](https://github.com/cristiandarioortegayubro/BDS/blob/main/images/Dendograma-001.png?raw=true)


## El algoritmo del clustering jerárquico


El algoritmo utilizado es muy simple. Primero hay que definir cual es la función de de disimilitud que se va utilizar. Luego se empieza fusionando las dos observaciones más similares, entonces quedan $n-1$ clústers. Luego se unen el otro par de grupos más similares, ahora quedan $n-2$. Así se continúa hasta que quedan todos en un solo grupo.


![Diagrama de clustering](https://github.com/cristiandarioortegayubro/BDS/blob/main/images/Dendograma-002.png?raw=true)


Sin embargo aún queda definir una función de disimilitud que funcione entre grupos. Esto se denomina enlace (*linkage*) y existen 3 formas distintas de enlace.

- **Enlace completo**: se toma como dismilitud entre grupos la máxima dismilitud:

\begin{equation}
d_{CL}(A,B)= max_{i∈A, j∈B}\left(d(i,j)\right)
\end{equation}

- **Enlace simple**: se considera la mínima disimilitud:

\begin{equation}
d_{CL}(A,B)= min_{i∈A, j∈B}\left(d(i,j)\right)
\end{equation}

- **Enlace promedio**: se computa el promedio de todas las disimilutdes:

\begin{equation}
d_{AL}(A,B)= \frac{1}{\left|A\right|\left|B\right|}\sum_{i∈A}\sum_{j∈B}d(i,J)
\end{equation}

Existe otra forma de enlace que es la denominada **Ward**. Esta va evaluando la distancia de cada observación al centroide del grupo y va evaluando como se va empeorando esta distancia a medida que se agregan grupos.


In [ ]:
house = house[['housing_median_age','median_house_value']][:30]
house.head()

Para poder realizar un agrupamiento jerárquico lo primero que necesitamos hacer es calcular las distancias que existen entre los puntos.

Para hacerlo, vamos a utilizar la función `pdist` de la biblioteca `scipy`. La función recibe como parámetros un `array` o un `DataFrame` de $n \times m$, y una métrica que es la que nos va definir la función distancia que se va utilizar.

El resultado obtenido va ser un `array` de un largo de $\frac{n(n-1)}{2}$.

La función `squareform` de `scipy` convierte el array en una matriz cuadrada $n\times n$ con las distancias calculadas.


In [ ]:
from scipy.spatial.distance import pdist, squareform
distancias = pd.DataFrame(squareform(pdist(house, metric = 'euclidean')), columns = house.index, index = house.index)
distancias.head()

Ahora nos queda ir agrupando las unidades de acuerdo a la similitud entre cada una de ellas. Para esto se utiliza lo que se denomina enlace (*linkage*).

También podemos utilizar para realizar estos enlaces una función de `scipy` denominada `linkage`.

Hiperparámetros:

 - `y`:  solicita una matriz de distancias condensada. Esto es la parte superior de una matriz triangular en un solo vector. Es el objeto que devuelve `pdist`. También se puede pasar una matriz con los datos, pero no un `DataFrame`.

- `mehod`: es el algoritmo de enlace a utilizar

- `metric`: es la métrica de distancia o disimilitud que se va utilizar.


Aplicamos la función con el método `linkage` usando como entrada la salida de `pdist`.


In [ ]:
from scipy.cluster.hierarchy import linkage

complete_clusters = linkage(pdist(house, metric = 'euclidean') #utilizamos la salida de la función pdist
                  , method = 'complete', metric = 'euclidean')

También podemos usar la matriz de valores $n\times m$ para calcularlo


In [ ]:
complete_clusters2 = linkage(house.values       #utilizamos los valores del dataframe en forma matricial
                             , method = 'complete', metric = 'euclidean')

(complete_clusters == complete_clusters2).all() #comprobamos que los resultados sean iguales

Para graficar el dendrograma podemos utilizar la función `dendrogram` que también es pertenceciente a la biblioteca `scipy`.


In [ ]:
from scipy.cluster.hierarchy import dendrogram

dendro = dendrogram(complete_clusters #le pasamos como entrada la salida que obtuvimos al aplicar el linkage
                    , labels = house.index # Se agregan los ínidices del data frame para saber que observaciones agrupa
                    )
plt.tight_layout()
plt.ylabel('Distancia euclídea')
plt.show()

In [ ]:
single_clusters = linkage(pdist(house, metric = 'euclidean'), method = 'single', metric = 'euclidean')
dendro = dendrogram(single_clusters, labels = house.index)
plt.tight_layout()
plt.ylabel('Distancia euclídea')
plt.show()

In [ ]:
average_clusters = linkage(pdist(house, metric = 'euclidean'), method = 'average', metric = 'euclidean')
dendro = dendrogram(average_clusters, labels = house.index)
plt.tight_layout()
plt.ylabel('Distancia euclídea')
plt.show()

In [ ]:
ward_clusters = linkage(pdist(house, metric = 'euclidean'), method = 'ward', metric = 'euclidean')
dendro = dendrogram(ward_clusters, labels = house.index)
plt.tight_layout()
plt.ylabel('Distancia euclídea')
plt.show()

# Conclusiones


En este colab nosotros:

- Aprendimos como funciona el algorítmo de K-means internamente.

- Elaboramos un programa que agrupa unidades de acuerdo al algoritmo K-means.

- Aprendimos como funciona un algorítmo de agrupación jerárquica.

- Utilizamos la biblioteca `scipy` para construir matrices de distancias y dendrogramas.
